In [ ]:
import os
import gc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from datasets import load_dataset
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import LabelEncoder
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoConfig,
    PreTrainedModel,
    Trainer,
    TrainingArguments,
    TrainerCallback
)

# Suppress tokenizer warnings
warnings.filterwarnings("ignore", category=UserWarning)

# ---------------------------------------------------------
# 1. CONFIGURATION
# ---------------------------------------------------------
SEEDS = [42, 123, 2024]
MODEL_CHECKPOINT = "microsoft/deberta-v3-base"
MAX_LENGTH = 512
BATCH_SIZE = 8
GRAD_ACCUMULATION = 4
LEARNING_RATE = 1.5e-5
EPOCHS = 8
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUTPUT_DIR = "./experiment_logs"

print(f"Using device: {DEVICE}")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---------------------------------------------------------
# 2. DATA PREPARATION & ENCODERS
# ---------------------------------------------------------
print("Loading dataset...")
dataset = load_dataset("ailsntua/QEvasion")
clarity_mapping = {"Clear Reply": 0, "Ambivalent": 1, "Clear Non-Reply": 2}
inv_clarity_mapping = {v: k for k, v in clarity_mapping.items()}

# --- A. Setup Individual Encoders ---
all_evasion_labels = []
for split in ['train', 'test']:
    for x in dataset[split]:
        if x['evasion_label']: all_evasion_labels.append(x['evasion_label'])
        for col in ['annotator1', 'annotator2', 'annotator3']:
            if x[col]: all_evasion_labels.append(x[col])

unique_evasion = sorted(list(set(all_evasion_labels)))
evasion_encoder = LabelEncoder()
evasion_encoder.fit(unique_evasion)
NUM_EVASION = len(unique_evasion)

# --- B. Setup Joint/Pair Encoder ---
legal_pairs_set = set()
for x in dataset['train']:
    c = x['clarity_label']
    e = x['evasion_label']
    if c and e:
        legal_pairs_set.add((c, e))

legal_pairs_list = sorted(list(legal_pairs_set))
joint_labels_str = [f"{c} | {e}" for c, e in legal_pairs_list]

joint_encoder = LabelEncoder()
joint_encoder.fit(joint_labels_str)
NUM_JOINT_LABELS = len(joint_labels_str)

print(f"✅ Found {NUM_JOINT_LABELS} unique legal pairs in Training set.")

# --- C. Create Constraints Mask ---
constraint_mask = torch.zeros((3, NUM_EVASION)).to(DEVICE)
for (c_str, e_str) in legal_pairs_list:
    c_idx = clarity_mapping[c_str]
    e_idx = evasion_encoder.transform([e_str])[0]
    constraint_mask[c_idx, e_idx] = 1.0

# ---------------------------------------------------------
# 3. PREPROCESSING
# ---------------------------------------------------------
def get_train_label(example):
    if example.get("evasion_label") in unique_evasion:
        return example["evasion_label"]
    votes = []
    for col in ['annotator1', 'annotator2', 'annotator3']:
        val = example.get(col)
        if val in unique_evasion: votes.append(val)
    if not votes: return None
    return Counter(votes).most_common(1)[0][0]

def preprocess(example):
    c_str = example["clarity_label"]
    e_str = get_train_label(example)
    
    example["labels_clarity"] = clarity_mapping.get(c_str, -1)
    if e_str:
        example["labels_evasion"] = int(evasion_encoder.transform([e_str])[0])
    else:
        example["labels_evasion"] = -100
        
    if c_str and e_str and (c_str, e_str) in legal_pairs_set:
        joint_str = f"{c_str} | {e_str}"
        example["labels_joint"] = int(joint_encoder.transform([joint_str])[0])
    else:
        example["labels_joint"] = -100
        
    return example

dataset = dataset.map(preprocess)
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

def tokenize(batch):
    return tokenizer(batch["interview_question"], batch["interview_answer"], 
                     padding="max_length", truncation=True, max_length=MAX_LENGTH)

tokenized_ds = dataset.map(tokenize, batched=True)
tokenized_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels_clarity", "labels_evasion", "labels_joint"])

# ---------------------------------------------------------
# 4. CALCULATE WEIGHTS (JOINT & INDEPENDENT)
# ---------------------------------------------------------
print("Calculating Class Weights...")

# --- 1. Independent Weights ---
y_train_e = np.array(tokenized_ds["train"]["labels_evasion"])
y_train_e = y_train_e[y_train_e != -100]
classes_e = np.arange(NUM_EVASION)
w_evasion = compute_class_weight('balanced', classes=classes_e, y=y_train_e)

# Manual Boost for Independent
partial_idx = evasion_encoder.transform(["Partial/half-answer"])[0]
w_evasion[partial_idx] = w_evasion[partial_idx] * 2.5
tensor_w_evasion = torch.tensor(w_evasion, dtype=torch.float).to(DEVICE)

# Clarity Weights
y_train_c = np.array(tokenized_ds["train"]["labels_clarity"])
y_train_c = y_train_c[y_train_c != -1]
classes_c = np.arange(3)
w_clarity = compute_class_weight('balanced', classes=classes_c, y=y_train_c)
tensor_w_clarity = torch.tensor(w_clarity, dtype=torch.float).to(DEVICE)

# --- 2. Joint Weights (NEW) ---
y_train_j = np.array(tokenized_ds["train"]["labels_joint"])
y_train_j = y_train_j[y_train_j != -100]
classes_j = np.arange(NUM_JOINT_LABELS)
w_joint = compute_class_weight('balanced', classes=classes_j, y=y_train_j)

# Manual Boost for Joint: Boost any pair that contains "Partial/half-answer"
print("Boosting Joint classes containing 'Partial/half-answer'...")
for i, label_str in enumerate(joint_encoder.classes_):
    if "Partial/half-answer" in label_str:
        w_joint[i] = w_joint[i] * 2.5

tensor_w_joint = torch.tensor(w_joint, dtype=torch.float).to(DEVICE)
print("✅ Weights Calculated.")

# ---------------------------------------------------------
# 5. MODEL DEFINITIONS
# ---------------------------------------------------------
class JointModel(PreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.deberta = AutoModel.from_config(config)
        self.classifier = nn.Linear(config.hidden_size, NUM_JOINT_LABELS)
        # Apply Weighted Loss
        self.loss_fct = nn.CrossEntropyLoss(weight=tensor_w_joint)
        
    def forward(self, input_ids, attention_mask, labels_joint=None, **kwargs):
        outputs = self.deberta(input_ids=input_ids, attention_mask=attention_mask)
        emb = outputs.last_hidden_state[:, 0, :]
        logits = self.classifier(emb)
        
        loss = None
        if labels_joint is not None:
            loss = self.loss_fct(logits, labels_joint)
            
        return (loss, logits) if loss is not None else logits

class IndependentModel(PreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.deberta = AutoModel.from_config(config)
        self.cls_clarity = nn.Linear(config.hidden_size, 3)
        self.cls_evasion = nn.Linear(config.hidden_size, NUM_EVASION)
        
        self.loss_c = nn.CrossEntropyLoss(weight=tensor_w_clarity)
        self.loss_e = nn.CrossEntropyLoss(weight=tensor_w_evasion)
        
    def forward(self, input_ids, attention_mask, labels_clarity=None, labels_evasion=None, **kwargs):
        outputs = self.deberta(input_ids=input_ids, attention_mask=attention_mask)
        emb = outputs.last_hidden_state[:, 0, :]
        
        logits_c = self.cls_clarity(emb)
        logits_e = self.cls_evasion(emb)
        
        loss = None
        if labels_evasion is not None and labels_clarity is not None:
            l_e = self.loss_e(logits_e, labels_evasion)
            l_c = 0
            if (labels_clarity != -1).any():
                l_c = self.loss_c(logits_c, labels_clarity)
            loss = l_e + 0.5 * l_c
            
        return (loss, logits_c, logits_e) if loss is not None else (logits_c, logits_e)

# ---------------------------------------------------------
# 6. CUSTOM LOGGING CALLBACK
# ---------------------------------------------------------
class EpochLoggingCallback(TrainerCallback):
    def __init__(self, model_type, val_dataset):
        self.model_type = model_type
        self.val_dataset = val_dataset

    def on_epoch_end(self, args, state, control, **kwargs):
        trainer = kwargs['model'].trainer
        output = trainer.predict(trainer.eval_dataset)
        
        final_pairs = []
        
        if self.model_type == "JOINT":
            preds = np.argmax(output.predictions, axis=-1)
            for idx in preds:
                joint_str = joint_encoder.inverse_transform([idx])[0]
                c_str, e_str = joint_str.split(" | ")
                final_pairs.append((c_str, e_str))
                
        elif self.model_type == "CONSTRAINED":
            logits_c = torch.tensor(output.predictions[0]).to(DEVICE)
            logits_e = torch.tensor(output.predictions[1]).to(DEVICE)
            probs_c = F.softmax(logits_c, dim=-1)
            probs_e = F.softmax(logits_e, dim=-1)
            
            joint_probs = torch.bmm(probs_c.unsqueeze(2), probs_e.unsqueeze(1))
            masked_probs = joint_probs * constraint_mask.unsqueeze(0)
            
            B = masked_probs.shape[0]
            flat_probs = masked_probs.view(B, -1)
            max_indices = torch.argmax(flat_probs, dim=1).cpu().numpy()
            
            for idx in max_indices:
                c_idx = idx // NUM_EVASION
                e_idx = idx % NUM_EVASION
                c_str = inv_clarity_mapping[c_idx]
                e_str = evasion_encoder.inverse_transform([e_idx])[0]
                final_pairs.append((c_str, e_str))
                
        elif self.model_type == "INDEPENDENT":
            pred_c = np.argmax(output.predictions[0], axis=-1)
            pred_e = np.argmax(output.predictions[1], axis=-1)
            for pc, pe in zip(pred_c, pred_e):
                c_str = inv_clarity_mapping[pc]
                e_str = evasion_encoder.inverse_transform([pe])[0]
                final_pairs.append((c_str, e_str))

        y_true_e = []
        y_pred_e = []
        y_true_c = []
        y_pred_c = []
        illegal_count = 0
        
        for i, (pred_c, pred_e) in enumerate(final_pairs):
            raw = self.val_dataset[i]
            if (pred_c, pred_e) not in legal_pairs_set: illegal_count += 1
                
            y_pred_e.append(pred_e)
            valid_labels = []
            if raw.get('evasion_label') in unique_evasion: valid_labels.append(raw['evasion_label'])
            for col in ['annotator1', 'annotator2', 'annotator3']:
                if raw.get(col) in unique_evasion: valid_labels.append(raw[col])
            
            if pred_e in valid_labels: y_true_e.append(pred_e)
            else: y_true_e.append(valid_labels[0] if valid_labels else "Explicit")

            y_pred_c.append(pred_c)
            y_true_c.append(raw['clarity_label'] if raw['clarity_label'] else "Clear Reply")
                
        f1_e = f1_score(y_true_e, y_pred_e, average='macro')
        f1_c = f1_score(y_true_c, y_pred_c, average='macro')
        ill_pct = (illegal_count / len(final_pairs)) * 100
        val_loss = output.metrics.get('test_loss', 0.0)

        print(f"   Epoch {state.epoch:.0f} | Loss: {val_loss:.4f} | Evasion F1: {f1_e:.4f} | Clarity F1: {f1_c:.4f} | Illegal: {ill_pct:.2f}%")


# ---------------------------------------------------------
# 7. MAIN EXPERIMENT LOOP
# ---------------------------------------------------------
EXPERIMENTS = [
    ("JOINT", JointModel),
    ("CONSTRAINED", IndependentModel),
    ("INDEPENDENT", IndependentModel)
]

all_results = []
cm_data_store = {} 

print(f"🚀 Starting Experiment (Epochs={EPOCHS}, Seeds={SEEDS})")

for model_name, model_class in EXPERIMENTS:
    print(f"\n{'='*20} RUNNING MODEL: {model_name} {'='*20}")
    
    for seed in SEEDS:
        print(f"--- Seed {seed} ---")
        
        torch.manual_seed(seed)
        np.random.seed(seed)
        if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
        
        config = AutoConfig.from_pretrained(MODEL_CHECKPOINT)
        model = model_class.from_pretrained(MODEL_CHECKPOINT, config=config).to(DEVICE)
        for param in model.deberta.embeddings.parameters(): param.requires_grad = False
        
        logging_cb = EpochLoggingCallback(model_type=model_name, val_dataset=dataset['test'])

        args = TrainingArguments(
            output_dir=f"{OUTPUT_DIR}/{model_name}_{seed}",
            learning_rate=LEARNING_RATE,
            per_device_train_batch_size=BATCH_SIZE,
            gradient_accumulation_steps=GRAD_ACCUMULATION,
            num_train_epochs=EPOCHS,
            eval_strategy="no",
            save_strategy="no",
            logging_steps=5000,
            report_to="none",
            seed=seed
        )
        
        trainer = Trainer(
            model=model,
            args=args,
            train_dataset=tokenized_ds["train"],
            eval_dataset=tokenized_ds["test"],
            tokenizer=tokenizer,
            callbacks=[logging_cb]
        )
        model.trainer = trainer 
        
        trainer.train()
        
        # --- FINAL COLLECTION ---
        output = trainer.predict(tokenized_ds['test'])
        final_pairs = []
        if model_name == "JOINT":
            preds = np.argmax(output.predictions, axis=-1)
            for idx in preds:
                joint_str = joint_encoder.inverse_transform([idx])[0]
                c_str, e_str = joint_str.split(" | ")
                final_pairs.append((c_str, e_str))
        elif model_name == "CONSTRAINED":
            logits_c = torch.tensor(output.predictions[0]).to(DEVICE)
            logits_e = torch.tensor(output.predictions[1]).to(DEVICE)
            probs_c = F.softmax(logits_c, dim=-1)
            probs_e = F.softmax(logits_e, dim=-1)
            joint_probs = torch.bmm(probs_c.unsqueeze(2), probs_e.unsqueeze(1))
            masked_probs = joint_probs * constraint_mask.unsqueeze(0)
            B = masked_probs.shape[0]
            flat_probs = masked_probs.view(B, -1)
            max_indices = torch.argmax(flat_probs, dim=1).cpu().numpy()
            for idx in max_indices:
                c_idx = idx // NUM_EVASION
                e_idx = idx % NUM_EVASION
                c_str = inv_clarity_mapping[c_idx]
                e_str = evasion_encoder.inverse_transform([e_idx])[0]
                final_pairs.append((c_str, e_str))
        elif model_name == "INDEPENDENT":
            pred_c = np.argmax(output.predictions[0], axis=-1)
            pred_e = np.argmax(output.predictions[1], axis=-1)
            for pc, pe in zip(pred_c, pred_e):
                c_str = inv_clarity_mapping[pc]
                e_str = evasion_encoder.inverse_transform([pe])[0]
                final_pairs.append((c_str, e_str))

        y_true_aligned = []
        y_pred_evasion = []
        illegal_count = 0
        joint_true_cm = []
        joint_pred_cm = []
        
        for i, (pred_c, pred_e) in enumerate(final_pairs):
            raw = dataset['test'][i]
            if (pred_c, pred_e) not in legal_pairs_set: illegal_count += 1
            y_pred_evasion.append(pred_e)
            valid_labels = []
            if raw.get('evasion_label') in unique_evasion: valid_labels.append(raw['evasion_label'])
            for col in ['annotator1', 'annotator2', 'annotator3']:
                if raw.get(col) in unique_evasion: valid_labels.append(raw[col])
            
            if pred_e in valid_labels: y_true_aligned.append(pred_e)
            else: y_true_aligned.append(valid_labels[0] if valid_labels else "Explicit")
            
            gt_c = raw['clarity_label']
            gt_e = raw['evasion_label'] if raw['evasion_label'] else (valid_labels[0] if valid_labels else "Explicit")
            if gt_c and gt_e and (gt_c, gt_e) in legal_pairs_set:
                joint_true_cm.append(f"{gt_c} | {gt_e}")
                joint_pred_cm.append(f"{pred_c} | {pred_e}")

        res = {
            "F1 Score": f1_score(y_true_aligned, y_pred_evasion, average='macro'),
            "Accuracy": accuracy_score(y_true_aligned, y_pred_evasion),
            "Precision": precision_score(y_true_aligned, y_pred_evasion, average='macro', zero_division=0),
            "Recall": recall_score(y_true_aligned, y_pred_evasion, average='macro', zero_division=0),
            "Illegal %": (illegal_count / len(final_pairs)) * 100,
            "Model": model_name,
            "Seed": seed
        }
        all_results.append(res)
        
        if seed == SEEDS[-1]:
            cm_data_store[model_name] = (joint_true_cm, joint_pred_cm)
            
        del model, trainer
        torch.cuda.empty_cache()
        gc.collect()

# ---------------------------------------------------------
# 8. STATISTICS REPORT
# ---------------------------------------------------------
df = pd.DataFrame(all_results)

print("\n\n" + "#"*60)
print("FINAL COMPARATIVE REPORT")
print("#"*60 + "\n")

cols_order = ['F1 Score', 'Accuracy', 'Precision', 'Recall', 'Illegal %']

for model_name in ["JOINT", "CONSTRAINED", "INDEPENDENT"]:
    sub_df = df[df['Model'] == model_name]
    stats = sub_df[cols_order].agg(['min', 'max', 'mean', 'std']).T
    
    print(f"\n⭐️ MODEL: {model_name}")
    print(f"{'Metric':<15} {'Min':<10} {'Max':<10} {'Avg ± Std':<20}")
    print("-" * 55)
    for index, row in stats.iterrows():
        avg_std = f"{row['mean']:.4f} ± {row['std']:.4f}"
        print(f"{index:<15} {row['min']:.4f}     {row['max']:.4f}     {avg_std}")

# ---------------------------------------------------------
# 9. PLOT CONFUSION MATRICES
# ---------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(24, 8))
plt.subplots_adjust(wspace=0.4)

for i, model_name in enumerate(["JOINT", "CONSTRAINED", "INDEPENDENT"]):
    true_labels, pred_labels = cm_data_store[model_name]
    unique_lbls = sorted(list(set(true_labels + pred_labels)))
    cm = confusion_matrix(true_labels, pred_labels, labels=unique_lbls)
    
    sns.heatmap(cm, annot=False, fmt='d', cmap='Blues', ax=axes[i], cbar=False)
    
    if len(unique_lbls) < 20:
        for r in range(len(unique_lbls)):
            for c in range(len(unique_lbls)):
                if cm[r, c] > 0:
                    axes[i].text(c+0.5, r+0.5, str(cm[r, c]), ha='center', va='center', fontsize=8)

    axes[i].set_title(f"{model_name}\n(Illegal Pairs in Red on Axis)")
    axes[i].set_xticklabels(unique_lbls, rotation=90, fontsize=8)
    axes[i].set_yticklabels(unique_lbls, rotation=0, fontsize=8)
    
    if model_name == "INDEPENDENT":
        xtick_objs = axes[i].get_xticklabels()
        for lbl_obj in xtick_objs:
            txt = lbl_obj.get_text()
            c_part, e_part = txt.split(" | ")
            if (c_part, e_part) not in legal_pairs_set:
                lbl_obj.set_color('red')
                lbl_obj.set_fontweight('bold')

plt.savefig(f"{OUTPUT_DIR}/final_comparison_cm.png")
print(f"\n✅ Confusion Matrices saved to {OUTPUT_DIR}/final_comparison_cm.png")